# Phase 12 — Controlled Model Challenger Benchmark

- **12A.** Experiment Contract & Safety
- **12B.** Champion Reproduction
- **12C.** Benchmark Utilities
- **12D.** Historical Controls
- **12E.** Focused 13–72 XGBoost
- **12F.** Focused XGBoost Variants
- **12G.** HistGradientBoosting
- **12H.** Random Forest
- **12I.** Stage-1 Leaderboard
- **12J.** External Challenger Gate
- **12K.** Robustness Analysis
- **12L.** Validation Selection Freeze
- **12M.** One-Time Final Test
- **12N.** Final Model Decision
- **12O.** Production Impact Plan
- **12P.** Save Benchmark Artifacts

## **12A.** Experiment Contract & Safety Validation

Before training any challenger, this phase verifies that the benchmark is
operating on the exact artifacts and data contract used by the existing
production model.

This phase validates:

- required experiment artifacts
- Phase 2 and production feature-contract agreement
- ordered model features
- target and identifier columns
- train, validation, and test schemas
- frozen split row counts and timestamp boundaries
- complete 1–72 hour forecast-horizon coverage
- missing-value constraints
- duplicate reference/horizon constraints
- chronological and purge-boundary integrity
- production model metadata consistency
- local software environment and artifact fingerprints

The test split is accessed here only for structural contract validation.
No test predictions, target-based test metrics, or model-selection decisions
are performed in this phase.

No model is trained and no production artifact is modified.

In [1]:
from __future__ import annotations

import hashlib
import json
import platform
import sys
from importlib.metadata import version
from pathlib import Path
from typing import Any

import joblib
import numpy as np
import pandas as pd
from IPython.display import display

In [2]:
def find_project_root(
    start: Path | None = None,
) -> Path:
    """Locate the repository root from the current notebook environment."""

    current = (start or Path.cwd()).resolve()

    for candidate in (current, *current.parents):
        if (
            (candidate / "pyproject.toml").exists()
            and (candidate / "data" / "training").exists()
            and (candidate / "models").exists()
        ):
            return candidate

    raise RuntimeError(
        "Could not locate the Pearls AQI Predictor project root."
    )


PROJECT_ROOT = find_project_root()

DATA_DIR = PROJECT_ROOT / "data" / "training"
MODEL_DIR = PROJECT_ROOT / "models"
REPORT_DIR = PROJECT_ROOT / "reports"

print("Project root:", PROJECT_ROOT)
print("Training data:", DATA_DIR)
print("Models:", MODEL_DIR)
print("Reports:", REPORT_DIR)

Project root: /home/riyan/Riyan/projects/pearls-aqi-predictor
Training data: /home/riyan/Riyan/projects/pearls-aqi-predictor/data/training
Models: /home/riyan/Riyan/projects/pearls-aqi-predictor/models
Reports: /home/riyan/Riyan/projects/pearls-aqi-predictor/reports


In [3]:
PHASE_2_FEATURE_CONTRACT_PATH = (
    DATA_DIR / "feature_columns.json"
)

PHASE_2_VALIDATION_REPORT_PATH = (
    DATA_DIR / "phase_2_validation_report.json"
)

TRAIN_DATASET_PATH = (
    DATA_DIR / "train_dataset.parquet"
)

VALIDATION_DATASET_PATH = (
    DATA_DIR / "validation_dataset.parquet"
)

TEST_DATASET_PATH = (
    DATA_DIR / "test_dataset.parquet"
)

PRODUCTION_MODEL_PATH = (
    MODEL_DIR / "best_model.joblib"
)

PRODUCTION_FEATURE_CONTRACT_PATH = (
    MODEL_DIR / "model_feature_columns.json"
)

PRODUCTION_METADATA_PATH = (
    MODEL_DIR / "model_metadata.json"
)

MODEL_SELECTION_REPORT_PATH = (
    MODEL_DIR / "model_selection_report.json"
)


REQUIRED_ARTIFACTS = {
    "phase_2_feature_contract": PHASE_2_FEATURE_CONTRACT_PATH,
    "phase_2_validation_report": PHASE_2_VALIDATION_REPORT_PATH,
    "train_dataset": TRAIN_DATASET_PATH,
    "validation_dataset": VALIDATION_DATASET_PATH,
    "test_dataset": TEST_DATASET_PATH,
    "production_model": PRODUCTION_MODEL_PATH,
    "production_feature_contract": PRODUCTION_FEATURE_CONTRACT_PATH,
    "production_metadata": PRODUCTION_METADATA_PATH,
    "model_selection_report": MODEL_SELECTION_REPORT_PATH,
}

In [4]:
artifact_status_df = pd.DataFrame(
    [
        {
            "artifact": name,
            "path": str(path.relative_to(PROJECT_ROOT)),
            "exists": path.exists(),
        }
        for name, path in REQUIRED_ARTIFACTS.items()
    ]
)

display(artifact_status_df)

missing_artifacts = [
    name
    for name, path in REQUIRED_ARTIFACTS.items()
    if not path.exists()
]

assert not missing_artifacts, (
    "Required benchmark artifacts are missing: "
    f"{missing_artifacts}"
)

print("All required benchmark artifacts exist.")

,artifact,path,exists
0,phase_2_feature_contract,data/training/feature_columns.json,True
1,phase_2_validation_report,data/training/phase_2_validation_report.json,True
2,train_dataset,data/training/train_dataset.parquet,True
3,validation_dataset,data/training/validation_dataset.parquet,True
4,test_dataset,data/training/test_dataset.parquet,True
5,production_model,models/best_model.joblib,True
6,production_feature_contract,models/model_feature_columns.json,True
7,production_metadata,models/model_metadata.json,True
8,model_selection_report,models/model_selection_report.json,True


All required benchmark artifacts exist.


In [5]:
def load_json_object(
    path: Path,
) -> dict[str, Any]:
    """Load one JSON artifact and require a JSON object."""

    payload = json.loads(
        path.read_text(encoding="utf-8")
    )

    if not isinstance(payload, dict):
        raise ValueError(
            f"{path} must contain a JSON object."
        )

    return payload


phase_2_feature_contract = load_json_object(
    PHASE_2_FEATURE_CONTRACT_PATH
)

phase_2_validation_report = load_json_object(
    PHASE_2_VALIDATION_REPORT_PATH
)

production_feature_contract = load_json_object(
    PRODUCTION_FEATURE_CONTRACT_PATH
)

production_metadata = load_json_object(
    PRODUCTION_METADATA_PATH
)

model_selection_report = load_json_object(
    MODEL_SELECTION_REPORT_PATH
)

print("Experiment metadata loaded successfully.")

Experiment metadata loaded successfully.


In [6]:
MODEL_FEATURE_COLUMNS = list(
    phase_2_feature_contract["feature_columns"]
)

TARGET_COLUMN = str(
    phase_2_feature_contract["target_column"]
)

IDENTIFIER_COLUMNS = list(
    phase_2_feature_contract["identifier_columns"]
)

FORECAST_HORIZON_COLUMN = (
    "forecast_horizon_hours"
)

EXPECTED_HORIZONS = list(
    range(
        int(
            phase_2_feature_contract[
                "forecast_horizon_min"
            ]
        ),
        int(
            phase_2_feature_contract[
                "forecast_horizon_max"
            ]
        )
        + 1,
    )
)


print("Model features:", len(MODEL_FEATURE_COLUMNS))
print("Target:", TARGET_COLUMN)
print("Identifiers:", IDENTIFIER_COLUMNS)
print(
    "Forecast horizons:",
    EXPECTED_HORIZONS[0],
    "to",
    EXPECTED_HORIZONS[-1],
)

Model features: 56
Target: target_pm25_ug_m3
Identifiers: ['reference_time', 'target_time']
Forecast horizons: 1 to 72


In [7]:
production_feature_columns = list(
    production_feature_contract["feature_columns"]
)

metadata_feature_columns = list(
    production_metadata["ordered_feature_names"]
)

assert MODEL_FEATURE_COLUMNS == production_feature_columns, (
    "Phase 2 and production feature contracts differ."
)

assert MODEL_FEATURE_COLUMNS == metadata_feature_columns, (
    "Phase 2 features and production metadata features differ."
)

assert len(MODEL_FEATURE_COLUMNS) == 56

assert (
    production_feature_contract["feature_count"]
    == 56
)

assert (
    production_metadata["input_feature_count"]
    == 56
)

assert (
    TARGET_COLUMN
    == production_feature_contract["target_column"]
    == production_metadata["target_column"]
)

assert (
    IDENTIFIER_COLUMNS
    == production_feature_contract["identifier_columns"]
)

assert (
    FORECAST_HORIZON_COLUMN
    in MODEL_FEATURE_COLUMNS
)

assert TARGET_COLUMN not in MODEL_FEATURE_COLUMNS

for identifier_column in IDENTIFIER_COLUMNS:
    assert identifier_column not in MODEL_FEATURE_COLUMNS


future_pm25_features = [
    column
    for column in MODEL_FEATURE_COLUMNS
    if column.startswith("target_pm25")
]

assert not future_pm25_features, (
    "Future PM2.5-derived model inputs detected: "
    f"{future_pm25_features}"
)

print("Feature-contract agreement validated.")

Feature-contract agreement validated.


In [18]:
train_df = pd.read_parquet(
    TRAIN_DATASET_PATH
)

validation_df = pd.read_parquet(
    VALIDATION_DATASET_PATH
)

test_df = pd.read_parquet(
    TEST_DATASET_PATH
)


dataset_summary_df = pd.DataFrame(
    [
        {
            "split": "train",
            "rows": len(train_df),
            "columns": len(train_df.columns),
        },
        {
            "split": "validation",
            "rows": len(validation_df),
            "columns": len(validation_df.columns),
        },
        {
            "split": "test",
            "rows": len(test_df),
            "columns": len(test_df.columns),
        },
    ]
)

display(dataset_summary_df)

,split,rows,columns
0,train,364798,59
1,validation,71256,59
2,test,76813,59


In [20]:
expected_splits = (
    phase_2_validation_report["splits"]
)

assert len(train_df) == int(
    expected_splits["train"]["rows"]
)

assert len(validation_df) == int(
    expected_splits["validation"]["rows"]
)

assert len(test_df) == int(
    expected_splits["test"]["rows"]
)


assert (
    train_df.columns.tolist()
    == validation_df.columns.tolist()
    == test_df.columns.tolist()
)

assert train_df.dtypes.equals(
    validation_df.dtypes
)

assert train_df.dtypes.equals(
    test_df.dtypes
)


required_columns = {
    *MODEL_FEATURE_COLUMNS,
    TARGET_COLUMN,
    *IDENTIFIER_COLUMNS,
}

missing_columns = sorted(
    required_columns.difference(
        train_df.columns
    )
)

assert not missing_columns, (
    f"Training schema is missing columns: {missing_columns}"
)

print("Frozen split sizes and schemas validated.")

Frozen split sizes and schemas validated.


In [21]:
def normalize_timestamp_columns(
    dataframe: pd.DataFrame,
) -> pd.DataFrame:
    """Return a copy with identifier timestamps normalized to UTC."""

    result = dataframe.copy()

    for column in IDENTIFIER_COLUMNS:
        result[column] = pd.to_datetime(
            result[column],
            utc=True,
            errors="raise",
        )

    return result


train_checked_df = normalize_timestamp_columns(
    train_df
)

validation_checked_df = normalize_timestamp_columns(
    validation_df
)

test_checked_df = normalize_timestamp_columns(
    test_df
)

print("Timestamp columns normalized for contract checks.")

Timestamp columns normalized for contract checks.


In [22]:
split_frames = {
    "train": train_checked_df,
    "validation": validation_checked_df,
    "test": test_checked_df,
}


integrity_records = []

for split_name, dataframe in split_frames.items():
    missing_features = int(
        dataframe[
            MODEL_FEATURE_COLUMNS
        ]
        .isna()
        .sum()
        .sum()
    )

    missing_targets = int(
        dataframe[
            TARGET_COLUMN
        ].isna().sum()
    )

    duplicate_keys = int(
        dataframe.duplicated(
            subset=[
                "reference_time",
                FORECAST_HORIZON_COLUMN,
            ]
        ).sum()
    )

    horizons = sorted(
        dataframe[
            FORECAST_HORIZON_COLUMN
        ]
        .astype(int)
        .unique()
        .tolist()
    )

    integrity_records.append(
        {
            "split": split_name,
            "missing_features": missing_features,
            "missing_targets": missing_targets,
            "duplicate_reference_horizon_keys": duplicate_keys,
            "horizon_min": min(horizons),
            "horizon_max": max(horizons),
            "unique_horizons": len(horizons),
        }
    )

    assert missing_features == 0
    assert missing_targets == 0
    assert duplicate_keys == 0
    assert horizons == EXPECTED_HORIZONS


integrity_df = pd.DataFrame(
    integrity_records
)

display(integrity_df)

print("Dataset integrity validation passed.")

,split,missing_features,missing_targets,duplicate_reference_horizon_keys,horizon_min,horizon_max,unique_horizons
0,train,0,0,0,1,72,72
1,validation,0,0,0,1,72,72
2,test,0,0,0,1,72,72


Dataset integrity validation passed.


In [23]:
split_boundary_df = pd.DataFrame(
    [
        {
            "split": "train",
            "reference_start": (
                train_checked_df["reference_time"].min()
            ),
            "reference_end": (
                train_checked_df["reference_time"].max()
            ),
            "target_start": (
                train_checked_df["target_time"].min()
            ),
            "target_end": (
                train_checked_df["target_time"].max()
            ),
        },
        {
            "split": "validation",
            "reference_start": (
                validation_checked_df["reference_time"].min()
            ),
            "reference_end": (
                validation_checked_df["reference_time"].max()
            ),
            "target_start": (
                validation_checked_df["target_time"].min()
            ),
            "target_end": (
                validation_checked_df["target_time"].max()
            ),
        },
        {
            "split": "test",
            "reference_start": (
                test_checked_df["reference_time"].min()
            ),
            "reference_end": (
                test_checked_df["reference_time"].max()
            ),
            "target_start": (
                test_checked_df["target_time"].min()
            ),
            "target_end": (
                test_checked_df["target_time"].max()
            ),
        },
    ]
)

display(split_boundary_df)

,split,reference_start,reference_end,target_start,target_end
0,train,2025-07-09 00:00:00+00:00,2026-03-18 11:00:00+00:00,2025-07-09 01:00:00+00:00,2026-03-21 11:00:00+00:00
1,validation,2026-03-21 12:00:00+00:00,2026-05-27 21:00:00+00:00,2026-03-21 13:00:00+00:00,2026-05-30 21:00:00+00:00
2,test,2026-05-30 22:00:00+00:00,2026-07-23 22:00:00+00:00,2026-05-30 23:00:00+00:00,2026-07-23 23:00:00+00:00


In [24]:
for split_name, dataframe in split_frames.items():
    expected = expected_splits[split_name]

    assert (
        dataframe["reference_time"].min()
        == pd.Timestamp(
            expected["reference_start"]
        )
    )

    assert (
        dataframe["reference_time"].max()
        == pd.Timestamp(
            expected["reference_end"]
        )
    )

    assert (
        dataframe["target_time"].min()
        == pd.Timestamp(
            expected["target_start"]
        )
    )

    assert (
        dataframe["target_time"].max()
        == pd.Timestamp(
            expected["target_end"]
        )
    )


assert (
    train_checked_df["target_time"].max()
    < validation_checked_df["reference_time"].min()
)

assert (
    validation_checked_df["target_time"].max()
    < test_checked_df["reference_time"].min()
)


for dataframe in split_frames.values():
    assert (
        dataframe["target_time"]
        > dataframe["reference_time"]
    ).all()


print("Chronological split and purge boundaries validated.")

Chronological split and purge boundaries validated.


In [25]:
EXPECTED_STRATEGY = (
    "hybrid_persistence_1_12_"
    "xgboost_shallower_13_72"
)

PERSISTENCE_MAX_HORIZON = 12


assert (
    production_metadata["selected_strategy"]
    == EXPECTED_STRATEGY
)

assert (
    model_selection_report["selected_strategy"]
    == EXPECTED_STRATEGY
)

assert (
    production_metadata["model_name"]
    == "xgboost_shallower"
)

assert (
    model_selection_report["selected_model"]
    == "xgboost_shallower"
)

assert (
    int(
        production_metadata[
            "routing"
        ]["persistence_max_horizon"]
    )
    == PERSISTENCE_MAX_HORIZON
)

assert (
    production_metadata[
        "routing"
    ]["horizons_1_to_12"]
    == "current_pm25_persistence"
)

assert (
    production_metadata[
        "routing"
    ]["horizons_13_to_72"]
    == "xgboost_shallower"
)


print("Production strategy metadata validated.")

Production strategy metadata validated.


In [26]:
current_environment = {
    "python": platform.python_version(),
    "pandas": version("pandas"),
    "numpy": version("numpy"),
    "scikit_learn": version("scikit-learn"),
    "xgboost": version("xgboost"),
    "joblib": version("joblib"),
}


production_environment = (
    production_metadata.get(
        "software_versions",
        {}
    )
)


environment_comparison_df = pd.DataFrame(
    [
        {
            "package": package,
            "current": current_environment.get(
                package
            ),
            "production_training": (
                production_environment.get(
                    package
                )
            ),
        }
        for package in current_environment
    ]
)

display(environment_comparison_df)

,package,current,production_training
0,python,3.12.3,3.12.3
1,pandas,2.3.3,3.0.5
2,numpy,2.2.6,2.5.1
3,scikit_learn,1.9.0,1.9.0
4,xgboost,3.3.0,3.3.0
5,joblib,1.5.3,None


In [27]:
def calculate_sha256(
    path: Path,
    chunk_size: int = 1024 * 1024,
) -> str:
    """Calculate the SHA-256 checksum of a local artifact."""

    digest = hashlib.sha256()

    with path.open("rb") as file:
        while chunk := file.read(chunk_size):
            digest.update(chunk)

    return digest.hexdigest()


fingerprint_paths = {
    "train_dataset": TRAIN_DATASET_PATH,
    "validation_dataset": VALIDATION_DATASET_PATH,
    "test_dataset": TEST_DATASET_PATH,
    "phase_2_feature_contract": (
        PHASE_2_FEATURE_CONTRACT_PATH
    ),
    "production_feature_contract": (
        PRODUCTION_FEATURE_CONTRACT_PATH
    ),
    "production_model": PRODUCTION_MODEL_PATH,
    "production_metadata": PRODUCTION_METADATA_PATH,
}


artifact_fingerprints = {
    name: calculate_sha256(path)
    for name, path in fingerprint_paths.items()
}


fingerprint_df = pd.DataFrame(
    [
        {
            "artifact": name,
            "sha256": checksum,
        }
        for name, checksum
        in artifact_fingerprints.items()
    ]
)

display(fingerprint_df)

,artifact,sha256
0,train_dataset,aa8bd58e0a1dc3dbc7e35c1dd4216d5400da22ef26d656...
1,validation_dataset,b30b259d1daeec75386a0b9dfb81fa8d90deaaa22d86e3...
2,test_dataset,0359ff647d1bfa3f021ca8e0ae6a2313a4831addb09a5d...
3,phase_2_feature_contract,21260e0ab94d9196ff5fa4d782517cf113d61f8eb578c2...
4,production_feature_contract,613b785c93a2c4135416906cb4a1c09b402c53df68674e...
5,production_model,9d1cbb032a6f376892c2ad047bbf93bf485d18b21b47d1...
6,production_metadata,11a398dec9abdc619d25db46b3065ae400db4cfb9df49b...


In [28]:
contract_validation_summary = {
    "status": "PASSED",
    "feature_count": len(
        MODEL_FEATURE_COLUMNS
    ),
    "target_column": TARGET_COLUMN,
    "identifier_columns": (
        IDENTIFIER_COLUMNS
    ),
    "forecast_horizon_min": min(
        EXPECTED_HORIZONS
    ),
    "forecast_horizon_max": max(
        EXPECTED_HORIZONS
    ),
    "train_rows": len(train_df),
    "validation_rows": len(
        validation_df
    ),
    "test_rows": len(test_df),
    "selected_strategy": (
        production_metadata[
            "selected_strategy"
        ]
    ),
    "production_model_name": (
        production_metadata["model_name"]
    ),
    "persistence_max_horizon": (
        PERSISTENCE_MAX_HORIZON
    ),
}


display(
    pd.Series(
        contract_validation_summary,
        name="value",
    ).to_frame()
)

print(
    "Phase 12A PASSED — frozen experiment "
    "contract is internally consistent."
)

,value
status,PASSED
feature_count,56
target_column,target_pm25_ug_m3
identifier_columns,"[reference_time, target_time]"
forecast_horizon_min,1
forecast_horizon_max,72
train_rows,364798
validation_rows,71256
test_rows,76813
selected_strategy,hybrid_persistence_1_12_xgboost_shallower_13_72


Phase 12A PASSED — frozen experiment contract is internally consistent.


## **12B.** Production Champion Reproduction

Before introducing any challenger, this phase independently reproduces the
existing production champion on the frozen validation split.

The current production strategy is:

- horizons 1–12: current-value PM2.5 persistence
- horizons 13–72: `xgboost_shallower`

The saved production model is loaded without retraining. Predictions are
generated against the existing ordered feature contract, and the resulting
hybrid validation metrics are compared with the immutable metrics stored in
the production model metadata.

This is a hard benchmark gate.

If the saved validation metrics cannot be reproduced within a small numerical
tolerance, challenger development stops until the discrepancy is understood.

The historical test split remains excluded from performance evaluation.

In [29]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)


HORIZON_GROUPS = {
    "1-6h": (1, 6),
    "7-12h": (7, 12),
    "13-24h": (13, 24),
    "25-48h": (25, 48),
    "49-72h": (49, 72),
}


def calculate_regression_metrics(
    y_true: pd.Series | np.ndarray,
    y_pred: pd.Series | np.ndarray,
) -> dict[str, float]:
    """Calculate standard PM2.5 regression metrics."""

    actual = np.asarray(
        y_true,
        dtype="float64",
    )

    predicted = np.asarray(
        y_pred,
        dtype="float64",
    )

    return {
        "mae": float(
            mean_absolute_error(
                actual,
                predicted,
            )
        ),
        "rmse": float(
            np.sqrt(
                mean_squared_error(
                    actual,
                    predicted,
                )
            )
        ),
        "r2": float(
            r2_score(
                actual,
                predicted,
            )
        ),
    }

In [30]:
production_model = joblib.load(
    PRODUCTION_MODEL_PATH
)


model_feature_count = getattr(
    production_model,
    "n_features_in_",
    None,
)


assert model_feature_count == len(
    MODEL_FEATURE_COLUMNS
), (
    "Saved production model feature count "
    "does not match the frozen contract."
)


loaded_model_type = type(
    production_model
).__name__


print(
    "Loaded model type:",
    loaded_model_type,
)

print(
    "Expected model type:",
    production_metadata["model_type"],
)

print(
    "Feature count:",
    model_feature_count,
)


assert (
    loaded_model_type
    == production_metadata["model_type"]
)

print("Saved production champion validated.")

Loaded model type: XGBRegressor
Expected model type: XGBRegressor
Feature count: 56
Saved production champion validated.


In [31]:
X_validation = validation_df[
    MODEL_FEATURE_COLUMNS
].copy()

y_validation = validation_df[
    TARGET_COLUMN
].astype(float).copy()


assert (
    X_validation.columns.tolist()
    == MODEL_FEATURE_COLUMNS
)

assert X_validation.isna().sum().sum() == 0

assert y_validation.isna().sum() == 0

assert np.isfinite(
    X_validation.to_numpy(
        dtype="float64"
    )
).all()

assert np.isfinite(
    y_validation.to_numpy(
        dtype="float64"
    )
).all()


validation_horizons = pd.to_numeric(
    validation_df[
        FORECAST_HORIZON_COLUMN
    ],
    errors="raise",
).astype(int)


print(
    "Validation rows:",
    len(validation_df),
)

print(
    "Validation horizon range:",
    validation_horizons.min(),
    "to",
    validation_horizons.max(),
)

Validation rows: 71256
Validation horizon range: 1 to 72


In [32]:
def generate_evaluation_hybrid_predictions(
    *,
    dataframe: pd.DataFrame,
    model: Any,
    feature_columns: list[str],
    persistence_max_horizon: int,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Generate production-equivalent hybrid predictions on a historical split.

    Returns both raw and operationally clipped predictions.
    """

    horizons = pd.to_numeric(
        dataframe[
            FORECAST_HORIZON_COLUMN
        ],
        errors="raise",
    )

    raw_predictions = np.empty(
        len(dataframe),
        dtype="float64",
    )

    persistence_mask = (
        horizons
        .le(persistence_max_horizon)
        .to_numpy()
    )

    model_mask = ~persistence_mask

    raw_predictions[
        persistence_mask
    ] = (
        dataframe.loc[
            persistence_mask,
            "pm25_current",
        ]
        .astype(float)
        .to_numpy()
    )

    if model_mask.any():
        raw_predictions[
            model_mask
        ] = model.predict(
            dataframe.loc[
                model_mask,
                feature_columns,
            ]
        )

    if not np.isfinite(
        raw_predictions
    ).all():
        raise ValueError(
            "Hybrid predictions contain "
            "NaN or infinite values."
        )

    operational_predictions = np.clip(
        raw_predictions,
        a_min=0.0,
        a_max=None,
    )

    return (
        raw_predictions,
        operational_predictions,
    )


(
    champion_validation_predictions_raw,
    champion_validation_predictions,
) = generate_evaluation_hybrid_predictions(
    dataframe=validation_df,
    model=production_model,
    feature_columns=MODEL_FEATURE_COLUMNS,
    persistence_max_horizon=(
        PERSISTENCE_MAX_HORIZON
    ),
)

In [33]:
persistence_mask = (
    validation_horizons
    .le(PERSISTENCE_MAX_HORIZON)
    .to_numpy()
)

model_mask = ~persistence_mask


expected_persistence_values = (
    validation_df.loc[
        persistence_mask,
        "pm25_current",
    ]
    .astype(float)
    .to_numpy()
)


np.testing.assert_allclose(
    champion_validation_predictions_raw[
        persistence_mask
    ],
    expected_persistence_values,
    rtol=0.0,
    atol=0.0,
)


assert persistence_mask.sum() > 0
assert model_mask.sum() > 0


print(
    "Persistence rows:",
    int(persistence_mask.sum()),
)

print(
    "Learned-model rows:",
    int(model_mask.sum()),
)

print(
    "Negative raw predictions:",
    int(
        (
            champion_validation_predictions_raw
            < 0
        ).sum()
    ),
)

print("Hybrid routing validation passed.")

Persistence rows: 12319
Learned-model rows: 58937
Negative raw predictions: 0
Hybrid routing validation passed.


In [34]:
reproduced_validation_metrics = (
    calculate_regression_metrics(
        y_true=y_validation,
        y_pred=(
            champion_validation_predictions
        ),
    )
)


expected_validation_metrics = {
    key: float(value)
    for key, value
    in production_metadata[
        "validation_metrics"
    ].items()
    if key in {
        "mae",
        "rmse",
        "r2",
    }
}


validation_metric_comparison_df = pd.DataFrame(
    [
        {
            "metric": metric,
            "expected": (
                expected_validation_metrics[
                    metric
                ]
            ),
            "reproduced": (
                reproduced_validation_metrics[
                    metric
                ]
            ),
            "absolute_difference": abs(
                reproduced_validation_metrics[
                    metric
                ]
                - expected_validation_metrics[
                    metric
                ]
            ),
        }
        for metric in (
            "mae",
            "rmse",
            "r2",
        )
    ]
)


display(
    validation_metric_comparison_df
)

,metric,expected,reproduced,absolute_difference
0,mae,6.695642,6.695642,2.664535e-15
1,rmse,9.419553,9.419553,0.000000e+00
2,r2,0.066611,0.066611,1.110223e-16


In [35]:
METRIC_ABSOLUTE_TOLERANCE = 1e-6


for metric_name in (
    "mae",
    "rmse",
    "r2",
):
    np.testing.assert_allclose(
        reproduced_validation_metrics[
            metric_name
        ],
        expected_validation_metrics[
            metric_name
        ],
        rtol=0.0,
        atol=METRIC_ABSOLUTE_TOLERANCE,
        err_msg=(
            "Production validation metric "
            f"could not be reproduced: "
            f"{metric_name}"
        ),
    )


print(
    "Production champion validation metrics "
    "reproduced successfully."
)

Production champion validation metrics reproduced successfully.


In [36]:
champion_validation_results_df = (
    validation_df[
        [
            "reference_time",
            "target_time",
            FORECAST_HORIZON_COLUMN,
            TARGET_COLUMN,
        ]
    ]
    .copy()
)


champion_validation_results_df[
    "prediction"
] = champion_validation_predictions


champion_horizon_group_records = []


for group_name, (
    minimum_horizon,
    maximum_horizon,
) in HORIZON_GROUPS.items():

    group_mask = (
        champion_validation_results_df[
            FORECAST_HORIZON_COLUMN
        ]
        .between(
            minimum_horizon,
            maximum_horizon,
        )
    )

    group_df = (
        champion_validation_results_df.loc[
            group_mask
        ]
    )

    group_metrics = (
        calculate_regression_metrics(
            y_true=group_df[
                TARGET_COLUMN
            ],
            y_pred=group_df[
                "prediction"
            ],
        )
    )

    champion_horizon_group_records.append(
        {
            "model": EXPECTED_STRATEGY,
            "horizon_group": group_name,
            "rows": len(group_df),
            **group_metrics,
        }
    )


champion_validation_by_horizon_group_df = (
    pd.DataFrame(
        champion_horizon_group_records
    )
)


display(
    champion_validation_by_horizon_group_df
)

,model,horizon_group,rows,mae,rmse,r2
0,hybrid_persistence_1_12_xgboost_shallower_13_72,1-6h,6190,3.648142,6.990142,0.516136
1,hybrid_persistence_1_12_xgboost_shallower_13_72,7-12h,6129,5.131832,8.653139,0.268671
2,hybrid_persistence_1_12_xgboost_shallower_13_72,13-24h,12163,7.117727,9.692964,0.091642
3,hybrid_persistence_1_12_xgboost_shallower_13_72,25-48h,23709,7.168849,9.669828,-0.050194
4,hybrid_persistence_1_12_xgboost_shallower_13_72,49-72h,23065,7.220054,9.769384,-0.031042


In [37]:
champion_per_horizon_records = []


for horizon in EXPECTED_HORIZONS:
    horizon_mask = (
        champion_validation_results_df[
            FORECAST_HORIZON_COLUMN
        ]
        .eq(horizon)
    )

    horizon_df = (
        champion_validation_results_df.loc[
            horizon_mask
        ]
    )

    horizon_metrics = (
        calculate_regression_metrics(
            y_true=horizon_df[
                TARGET_COLUMN
            ],
            y_pred=horizon_df[
                "prediction"
            ],
        )
    )

    champion_per_horizon_records.append(
        {
            "model": EXPECTED_STRATEGY,
            "forecast_horizon_hours": horizon,
            "rows": len(horizon_df),
            **horizon_metrics,
        }
    )


champion_validation_by_horizon_df = (
    pd.DataFrame(
        champion_per_horizon_records
    )
)


assert len(
    champion_validation_by_horizon_df
) == 72


display(
    champion_validation_by_horizon_df.head(
        12
    )
)

,model,forecast_horizon_hours,rows,mae,rmse,r2
0,hybrid_persistence_1_12_xgboost_shallower_13_72,1,1039,2.061886,4.209370,0.822803
1,hybrid_persistence_1_12_xgboost_shallower_13_72,2,1035,3.007246,5.920939,0.651224
2,hybrid_persistence_1_12_xgboost_shallower_13_72,3,1032,3.668605,7.005961,0.512925
3,hybrid_persistence_1_12_xgboost_shallower_13_72,4,1030,4.047670,7.627872,0.424880
4,hybrid_persistence_1_12_xgboost_shallower_13_72,5,1028,4.456031,8.173541,0.342357
5,hybrid_persistence_1_12_xgboost_shallower_13_72,6,1026,4.669883,8.170961,0.344593
6,hybrid_persistence_1_12_xgboost_shallower_13_72,7,1025,4.850439,8.232904,0.336435
7,hybrid_persistence_1_12_xgboost_shallower_13_72,8,1024,4.983105,8.450385,0.302254
8,hybrid_persistence_1_12_xgboost_shallower_13_72,9,1022,5.090802,8.522998,0.290635
9,hybrid_persistence_1_12_xgboost_shallower_13_72,10,1020,5.118431,8.521727,0.289935


In [38]:
learned_horizon_mask = (
    validation_horizons
    .gt(PERSISTENCE_MAX_HORIZON)
    .to_numpy()
)


champion_learned_horizon_metrics = (
    calculate_regression_metrics(
        y_true=(
            y_validation.to_numpy()[
                learned_horizon_mask
            ]
        ),
        y_pred=(
            champion_validation_predictions[
                learned_horizon_mask
            ]
        ),
    )
)


display(
    pd.Series(
        champion_learned_horizon_metrics,
        name="13-72h champion",
    ).to_frame()
)

,13-72h champion
mae,7.178338
rmse,9.713671
r2,-0.009030


In [39]:
champion_reproduction_summary = {
    "status": "PASSED",
    "model_type": (
        loaded_model_type
    ),
    "model_name": (
        production_metadata[
            "model_name"
        ]
    ),
    "strategy": (
        production_metadata[
            "selected_strategy"
        ]
    ),
    "persistence_max_horizon": (
        PERSISTENCE_MAX_HORIZON
    ),
    "validation_rows": len(
        validation_df
    ),
    "validation_mae": (
        reproduced_validation_metrics[
            "mae"
        ]
    ),
    "validation_rmse": (
        reproduced_validation_metrics[
            "rmse"
        ]
    ),
    "validation_r2": (
        reproduced_validation_metrics[
            "r2"
        ]
    ),
    "negative_raw_predictions": int(
        (
            champion_validation_predictions_raw
            < 0
        ).sum()
    ),
}


display(
    pd.Series(
        champion_reproduction_summary,
        name="value",
    ).to_frame()
)


print(
    "Phase 12B PASSED — the existing "
    "production champion has been reproduced."
)

,value
status,PASSED
model_type,XGBRegressor
model_name,xgboost_shallower
strategy,hybrid_persistence_1_12_xgboost_shallower_13_72
persistence_max_horizon,12
validation_rows,71256
validation_mae,6.695642
validation_rmse,9.419553
validation_r2,0.066611
negative_raw_predictions,0


Phase 12B PASSED — the existing production champion has been reproduced.


## **12C.** Standardized Benchmark Utilities

The production champion is now reproducible on the frozen validation split.

Before training any challenger, this phase defines a single evaluation contract
that will be applied consistently to every candidate model.

The utilities created here standardize:

- hybrid persistence/model prediction generation
- overall validation metrics
- learned-horizon metrics for hours 13–72
- horizon-group metrics
- individual forecast-horizon metrics
- severe-PM2.5 performance
- prediction validity and clipping behavior
- training and prediction runtime reporting
- serialized model size
- candidate result packaging

The utilities are estimator-neutral. They operate on a fitted estimator through
the standard `.predict()` interface and do not contain model-family-specific
training logic.

No challenger model is trained in this phase.

The historical test split remains excluded from performance evaluation.

In [40]:
from time import perf_counter
from tempfile import TemporaryDirectory

In [41]:
LEARNED_HORIZON_MIN = (
    PERSISTENCE_MAX_HORIZON + 1
)

LEARNED_HORIZON_MAX = max(
    EXPECTED_HORIZONS
)

SEVERE_PM25_THRESHOLD = 55.5


print(
    "Learned-model horizon range:",
    LEARNED_HORIZON_MIN,
    "to",
    LEARNED_HORIZON_MAX,
)

print(
    "Severe PM2.5 threshold:",
    SEVERE_PM25_THRESHOLD,
    "µg/m³",
)

Learned-model horizon range: 13 to 72
Severe PM2.5 threshold: 55.5 µg/m³


In [42]:
def calculate_horizon_group_metrics(
    *,
    dataframe: pd.DataFrame,
    prediction_column: str,
    model_name: str,
) -> pd.DataFrame:
    """Calculate regression metrics for each configured horizon group."""

    records: list[dict[str, Any]] = []

    for group_name, (
        minimum_horizon,
        maximum_horizon,
    ) in HORIZON_GROUPS.items():
        group_df = dataframe.loc[
            dataframe[
                FORECAST_HORIZON_COLUMN
            ].between(
                minimum_horizon,
                maximum_horizon,
            )
        ]

        if group_df.empty:
            raise ValueError(
                "No rows are available for "
                f"horizon group {group_name}."
            )

        metrics = calculate_regression_metrics(
            y_true=group_df[TARGET_COLUMN],
            y_pred=group_df[prediction_column],
        )

        records.append(
            {
                "model": model_name,
                "horizon_group": group_name,
                "horizon_min": minimum_horizon,
                "horizon_max": maximum_horizon,
                "rows": len(group_df),
                **metrics,
            }
        )

    result = pd.DataFrame(records)

    assert len(result) == len(
        HORIZON_GROUPS
    )

    return result

In [43]:
def calculate_per_horizon_metrics(
    *,
    dataframe: pd.DataFrame,
    prediction_column: str,
    model_name: str,
) -> pd.DataFrame:
    """Calculate regression metrics independently for horizons 1 through 72."""

    records: list[dict[str, Any]] = []

    observed_horizons = sorted(
        dataframe[
            FORECAST_HORIZON_COLUMN
        ]
        .astype(int)
        .unique()
        .tolist()
    )

    if observed_horizons != EXPECTED_HORIZONS:
        raise ValueError(
            "Evaluation data does not contain "
            "the complete expected horizon range."
        )

    for horizon in EXPECTED_HORIZONS:
        horizon_df = dataframe.loc[
            dataframe[
                FORECAST_HORIZON_COLUMN
            ].eq(horizon)
        ]

        if horizon_df.empty:
            raise ValueError(
                f"No rows found for horizon {horizon}."
            )

        metrics = calculate_regression_metrics(
            y_true=horizon_df[TARGET_COLUMN],
            y_pred=horizon_df[prediction_column],
        )

        records.append(
            {
                "model": model_name,
                "forecast_horizon_hours": horizon,
                "rows": len(horizon_df),
                **metrics,
            }
        )

    result = pd.DataFrame(records)

    assert len(result) == 72

    return result

In [44]:
def calculate_learned_horizon_metrics(
    *,
    dataframe: pd.DataFrame,
    prediction_column: str,
) -> dict[str, float]:
    """Calculate metrics only where the learned estimator is active."""

    learned_df = dataframe.loc[
        dataframe[
            FORECAST_HORIZON_COLUMN
        ].between(
            LEARNED_HORIZON_MIN,
            LEARNED_HORIZON_MAX,
        )
    ]

    if learned_df.empty:
        raise ValueError(
            "No learned-horizon evaluation rows are available."
        )

    return calculate_regression_metrics(
        y_true=learned_df[TARGET_COLUMN],
        y_pred=learned_df[prediction_column],
    )

In [45]:
def calculate_severe_pm25_metrics(
    *,
    dataframe: pd.DataFrame,
    prediction_column: str,
) -> dict[str, Any]:
    """Evaluate predictions for observations at or above the severe threshold."""

    severe_df = dataframe.loc[
        dataframe[
            TARGET_COLUMN
        ].ge(SEVERE_PM25_THRESHOLD)
    ]

    sample_count = len(
        severe_df
    )

    if sample_count == 0:
        return {
            "threshold_ug_m3": (
                SEVERE_PM25_THRESHOLD
            ),
            "sample_count": 0,
            "metrics": None,
        }

    metrics = calculate_regression_metrics(
        y_true=severe_df[TARGET_COLUMN],
        y_pred=severe_df[prediction_column],
    )

    return {
        "threshold_ug_m3": (
            SEVERE_PM25_THRESHOLD
        ),
        "sample_count": (
            sample_count
        ),
        "metrics": metrics,
    }

In [46]:
def summarize_prediction_validity(
    *,
    raw_predictions: np.ndarray,
    operational_predictions: np.ndarray,
) -> dict[str, Any]:
    """Summarize numerical validity and PM2.5 clipping behavior."""

    raw = np.asarray(
        raw_predictions,
        dtype="float64",
    )

    operational = np.asarray(
        operational_predictions,
        dtype="float64",
    )

    if raw.shape != operational.shape:
        raise ValueError(
            "Raw and operational predictions "
            "have different shapes."
        )

    raw_nan_count = int(
        np.isnan(raw).sum()
    )

    raw_inf_count = int(
        np.isinf(raw).sum()
    )

    negative_raw_count = int(
        (raw < 0).sum()
    )

    clipped_count = int(
        (
            ~np.isclose(
                raw,
                operational,
                rtol=0.0,
                atol=0.0,
            )
        ).sum()
    )

    return {
        "prediction_count": int(
            len(raw)
        ),
        "raw_nan_count": (
            raw_nan_count
        ),
        "raw_inf_count": (
            raw_inf_count
        ),
        "negative_raw_count": (
            negative_raw_count
        ),
        "clipped_prediction_count": (
            clipped_count
        ),
        "raw_min": (
            float(np.nanmin(raw))
            if raw_nan_count < len(raw)
            else None
        ),
        "raw_max": (
            float(np.nanmax(raw))
            if raw_nan_count < len(raw)
            else None
        ),
        "operational_min": (
            float(
                np.nanmin(
                    operational
                )
            )
            if len(operational)
            else None
        ),
        "operational_max": (
            float(
                np.nanmax(
                    operational
                )
            )
            if len(operational)
            else None
        ),
    }

In [47]:
def measure_serialized_model_size_bytes(
    model: Any,
) -> int:
    """Measure joblib artifact size without keeping a permanent file."""

    with TemporaryDirectory(
        prefix="pearls-benchmark-"
    ) as temporary_directory:
        model_path = (
            Path(temporary_directory)
            / "model.joblib"
        )

        joblib.dump(
            model,
            model_path,
        )

        return int(
            model_path.stat().st_size
        )

In [48]:
def generate_timed_hybrid_predictions(
    *,
    dataframe: pd.DataFrame,
    model: Any,
    feature_columns: list[str],
    persistence_max_horizon: int,
) -> tuple[np.ndarray, np.ndarray, float]:
    """Generate hybrid predictions and record prediction runtime."""

    prediction_start = perf_counter()

    (
        raw_predictions,
        operational_predictions,
    ) = generate_evaluation_hybrid_predictions(
        dataframe=dataframe,
        model=model,
        feature_columns=feature_columns,
        persistence_max_horizon=(
            persistence_max_horizon
        ),
    )

    prediction_seconds = (
        perf_counter()
        - prediction_start
    )

    return (
        raw_predictions,
        operational_predictions,
        float(prediction_seconds),
    )

In [49]:
def build_evaluation_frame(
    *,
    dataframe: pd.DataFrame,
    raw_predictions: np.ndarray,
    operational_predictions: np.ndarray,
) -> pd.DataFrame:
    """Build one aligned evaluation table from a frozen split."""

    raw = np.asarray(
        raw_predictions,
        dtype="float64",
    )

    operational = np.asarray(
        operational_predictions,
        dtype="float64",
    )

    if (
        len(dataframe) != len(raw)
        or len(dataframe) != len(
            operational
        )
    ):
        raise ValueError(
            "Prediction lengths do not match "
            "the evaluation dataframe."
        )

    result = dataframe[
        [
            "reference_time",
            "target_time",
            FORECAST_HORIZON_COLUMN,
            TARGET_COLUMN,
            "pm25_current",
        ]
    ].copy()

    result[
        "prediction_raw"
    ] = raw

    result[
        "prediction"
    ] = operational

    result[
        "prediction_was_clipped"
    ] = ~np.isclose(
        raw,
        operational,
        rtol=0.0,
        atol=0.0,
    )

    return result

In [50]:
def evaluate_candidate_predictions(
    *,
    candidate_name: str,
    model_family: str,
    training_scope: str,
    dataframe: pd.DataFrame,
    raw_predictions: np.ndarray,
    operational_predictions: np.ndarray,
    training_seconds: float | None,
    prediction_seconds: float,
    model_size_bytes: int | None,
    parameters: dict[str, Any] | None = None,
) -> dict[str, Any]:
    """Evaluate one fitted candidate under the standard benchmark contract."""

    evaluation_df = build_evaluation_frame(
        dataframe=dataframe,
        raw_predictions=raw_predictions,
        operational_predictions=(
            operational_predictions
        ),
    )

    overall_metrics = (
        calculate_regression_metrics(
            y_true=evaluation_df[
                TARGET_COLUMN
            ],
            y_pred=evaluation_df[
                "prediction"
            ],
        )
    )

    learned_metrics = (
        calculate_learned_horizon_metrics(
            dataframe=evaluation_df,
            prediction_column="prediction",
        )
    )

    horizon_group_metrics = (
        calculate_horizon_group_metrics(
            dataframe=evaluation_df,
            prediction_column="prediction",
            model_name=candidate_name,
        )
    )

    per_horizon_metrics = (
        calculate_per_horizon_metrics(
            dataframe=evaluation_df,
            prediction_column="prediction",
            model_name=candidate_name,
        )
    )

    severe_pm25 = (
        calculate_severe_pm25_metrics(
            dataframe=evaluation_df,
            prediction_column="prediction",
        )
    )

    prediction_validity = (
        summarize_prediction_validity(
            raw_predictions=(
                raw_predictions
            ),
            operational_predictions=(
                operational_predictions
            ),
        )
    )

    return {
        "candidate_name": (
            candidate_name
        ),
        "model_family": (
            model_family
        ),
        "training_scope": (
            training_scope
        ),
        "parameters": (
            parameters or {}
        ),
        "training_seconds": (
            None
            if training_seconds is None
            else float(training_seconds)
        ),
        "prediction_seconds": float(
            prediction_seconds
        ),
        "model_size_bytes": (
            model_size_bytes
        ),
        "model_size_mib": (
            None
            if model_size_bytes is None
            else float(
                model_size_bytes
                / (1024 ** 2)
            )
        ),
        "overall_metrics": (
            overall_metrics
        ),
        "learned_horizon_metrics": (
            learned_metrics
        ),
        "horizon_group_metrics": (
            horizon_group_metrics
        ),
        "per_horizon_metrics": (
            per_horizon_metrics
        ),
        "severe_pm25": (
            severe_pm25
        ),
        "prediction_validity": (
            prediction_validity
        ),
        "evaluation_frame": (
            evaluation_df
        ),
    }

In [51]:
def build_leaderboard_row(
    result: dict[str, Any],
) -> dict[str, Any]:
    """Flatten headline candidate metrics into one leaderboard row."""

    overall = result[
        "overall_metrics"
    ]

    learned = result[
        "learned_horizon_metrics"
    ]

    validity = result[
        "prediction_validity"
    ]

    return {
        "candidate": (
            result["candidate_name"]
        ),
        "model_family": (
            result["model_family"]
        ),
        "training_scope": (
            result["training_scope"]
        ),
        "hybrid_mae": (
            overall["mae"]
        ),
        "hybrid_rmse": (
            overall["rmse"]
        ),
        "hybrid_r2": (
            overall["r2"]
        ),
        "learned_13_72_mae": (
            learned["mae"]
        ),
        "learned_13_72_rmse": (
            learned["rmse"]
        ),
        "learned_13_72_r2": (
            learned["r2"]
        ),
        "negative_raw_predictions": (
            validity[
                "negative_raw_count"
            ]
        ),
        "clipped_predictions": (
            validity[
                "clipped_prediction_count"
            ]
        ),
        "training_seconds": (
            result[
                "training_seconds"
            ]
        ),
        "prediction_seconds": (
            result[
                "prediction_seconds"
            ]
        ),
        "model_size_mib": (
            result[
                "model_size_mib"
            ]
        ),
    }

In [52]:
def calculate_improvement_percent(
    *,
    champion_value: float,
    candidate_value: float,
) -> float:
    """Return positive percentage when a lower-is-better metric improves."""

    if champion_value == 0:
        raise ValueError(
            "Champion metric must be non-zero "
            "for percentage comparison."
        )

    return float(
        (
            champion_value
            - candidate_value
        )
        / abs(champion_value)
        * 100.0
    )

In [53]:
benchmark_results: dict[
    str,
    dict[str, Any],
] = {}

In [54]:
(
    champion_raw_12c,
    champion_predictions_12c,
    champion_prediction_seconds_12c,
) = generate_timed_hybrid_predictions(
    dataframe=validation_df,
    model=production_model,
    feature_columns=MODEL_FEATURE_COLUMNS,
    persistence_max_horizon=(
        PERSISTENCE_MAX_HORIZON
    ),
)


champion_model_size_bytes = (
    measure_serialized_model_size_bytes(
        production_model
    )
)


champion_benchmark_result = (
    evaluate_candidate_predictions(
        candidate_name="production_champion_v1",
        model_family=loaded_model_type,
        training_scope=(
            "historical production artifact"
        ),
        dataframe=validation_df,
        raw_predictions=(
            champion_raw_12c
        ),
        operational_predictions=(
            champion_predictions_12c
        ),
        training_seconds=None,
        prediction_seconds=(
            champion_prediction_seconds_12c
        ),
        model_size_bytes=(
            champion_model_size_bytes
        ),
        parameters=(
            production_metadata.get(
                "model_parameters",
                {},
            )
        ),
    )
)


benchmark_results[
    "production_champion_v1"
] = champion_benchmark_result

In [55]:
utility_champion_metrics = (
    champion_benchmark_result[
        "overall_metrics"
    ]
)


for metric_name in (
    "mae",
    "rmse",
    "r2",
):
    np.testing.assert_allclose(
        utility_champion_metrics[
            metric_name
        ],
        reproduced_validation_metrics[
            metric_name
        ],
        rtol=0.0,
        atol=METRIC_ABSOLUTE_TOLERANCE,
        err_msg=(
            "12C utility evaluation differs "
            "from the Phase 12B champion baseline."
        ),
    )


np.testing.assert_allclose(
    champion_raw_12c,
    champion_validation_predictions_raw,
    rtol=0.0,
    atol=1e-12,
)


np.testing.assert_allclose(
    champion_predictions_12c,
    champion_validation_predictions,
    rtol=0.0,
    atol=1e-12,
)


print(
    "12C benchmark utilities reproduce "
    "the Phase 12B champion baseline."
)

12C benchmark utilities reproduce the Phase 12B champion baseline.


In [56]:
champion_leaderboard_df = pd.DataFrame(
    [
        build_leaderboard_row(
            champion_benchmark_result
        )
    ]
)


display(
    champion_leaderboard_df
)

,candidate,model_family,training_scope,hybrid_mae,hybrid_rmse,hybrid_r2,learned_13_72_mae,learned_13_72_rmse,learned_13_72_r2,negative_raw_predictions,clipped_predictions,training_seconds,prediction_seconds,model_size_mib
0,production_champion_v1,XGBRegressor,historical production artifact,6.695642,9.419553,0.066611,7.178338,9.713671,-0.00903,0,0,None,0.26926,0.645863


In [57]:
display(
    champion_benchmark_result[
        "horizon_group_metrics"
    ]
)

,model,horizon_group,horizon_min,horizon_max,rows,mae,rmse,r2
0,production_champion_v1,1-6h,1,6,6190,3.648142,6.990142,0.516136
1,production_champion_v1,7-12h,7,12,6129,5.131832,8.653139,0.268671
2,production_champion_v1,13-24h,13,24,12163,7.117727,9.692964,0.091642
3,production_champion_v1,25-48h,25,48,23709,7.168849,9.669828,-0.050194
4,production_champion_v1,49-72h,49,72,23065,7.220054,9.769384,-0.031042


In [58]:
display(
    champion_benchmark_result[
        "per_horizon_metrics"
    ].head(12)
)

,model,forecast_horizon_hours,rows,mae,rmse,r2
0,production_champion_v1,1,1039,2.061886,4.209370,0.822803
1,production_champion_v1,2,1035,3.007246,5.920939,0.651224
2,production_champion_v1,3,1032,3.668605,7.005961,0.512925
3,production_champion_v1,4,1030,4.047670,7.627872,0.424880
4,production_champion_v1,5,1028,4.456031,8.173541,0.342357
5,production_champion_v1,6,1026,4.669883,8.170961,0.344593
6,production_champion_v1,7,1025,4.850439,8.232904,0.336435
7,production_champion_v1,8,1024,4.983105,8.450385,0.302254
8,production_champion_v1,9,1022,5.090802,8.522998,0.290635
9,production_champion_v1,10,1020,5.118431,8.521727,0.289935


In [59]:
display(
    pd.Series(
        champion_benchmark_result[
            "severe_pm25"
        ],
        name="value",
    ).to_frame()
)

,value
threshold_ug_m3,55.5
sample_count,181
metrics,"{'mae': 49.37182242461989, 'rmse': 50.15318038..."


In [60]:
champion_prediction_validity = (
    champion_benchmark_result[
        "prediction_validity"
    ]
)


display(
    pd.Series(
        champion_prediction_validity,
        name="value",
    ).to_frame()
)


assert (
    champion_prediction_validity[
        "raw_nan_count"
    ]
    == 0
)

assert (
    champion_prediction_validity[
        "raw_inf_count"
    ]
    == 0
)


print(
    "Champion prediction validity "
    "checks passed."
)

,value
prediction_count,71256.000000
raw_nan_count,0.000000
raw_inf_count,0.000000
negative_raw_count,0.000000
clipped_prediction_count,0.000000
raw_min,0.100000
raw_max,99.575546
operational_min,0.100000
operational_max,99.575546


Champion prediction validity checks passed.


In [61]:
benchmark_utility_summary = {
    "status": "PASSED",
    "registered_results": len(
        benchmark_results
    ),
    "champion_registered": (
        "production_champion_v1"
        in benchmark_results
    ),
    "overall_metrics_match_12b": True,
    "horizon_group_count": len(
        champion_benchmark_result[
            "horizon_group_metrics"
        ]
    ),
    "per_horizon_count": len(
        champion_benchmark_result[
            "per_horizon_metrics"
        ]
    ),
    "severe_threshold_ug_m3": (
        SEVERE_PM25_THRESHOLD
    ),
    "learned_horizon_min": (
        LEARNED_HORIZON_MIN
    ),
    "learned_horizon_max": (
        LEARNED_HORIZON_MAX
    ),
}


display(
    pd.Series(
        benchmark_utility_summary,
        name="value",
    ).to_frame()
)


print(
    "Phase 12C PASSED — standardized "
    "benchmark utilities are ready."
)

,value
status,PASSED
registered_results,1
champion_registered,True
overall_metrics_match_12b,True
horizon_group_count,5
per_horizon_count,72
severe_threshold_ug_m3,55.5
learned_horizon_min,13
learned_horizon_max,72


Phase 12C PASSED — standardized benchmark utilities are ready.


## **12D.** Historical Control Reproduction

Before beginning new challenger experiments, this phase reproduces a focused
subset of the original Phase 3 model comparisons.

The purpose is not to reopen the original model-selection process or tune old
models. Instead, these controls verify that the current benchmark environment,
frozen datasets, feature contract, and metric utilities produce conclusions
consistent with the original experiment.

The historical controls are:

- current-value persistence
- previous-day persistence
- Ridge Regression with the original Phase 3 configuration
- Histogram Gradient Boosting with the original Phase 3 configuration
- the saved `xgboost_shallower` production estimator

Two evaluation views are kept separate:

1. **Historical raw-model reproduction**  
   Reproduces the way these baselines and estimators were originally measured
   across the full validation split.

2. **Current hybrid benchmark view**  
   For fitted learned estimators, applies the production routing contract:
   persistence for hours 1–12 and the estimator for hours 13–72.

No hyperparameter tuning is performed in this phase.

No candidate is selected for production, and the historical test split remains
excluded from performance evaluation.

In [62]:
from sklearn.ensemble import (
    HistGradientBoostingRegressor,
)
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

In [63]:
X_train = train_df[
    MODEL_FEATURE_COLUMNS
].copy()

y_train = train_df[
    TARGET_COLUMN
].astype(float).copy()


assert (
    X_train.columns.tolist()
    == MODEL_FEATURE_COLUMNS
)

assert X_train.isna().sum().sum() == 0
assert y_train.isna().sum() == 0

assert np.isfinite(
    X_train.to_numpy(
        dtype="float64"
    )
).all()

assert np.isfinite(
    y_train.to_numpy(
        dtype="float64"
    )
).all()


print(
    "Training rows:",
    len(X_train),
)

print(
    "Training features:",
    X_train.shape[1],
)

print(
    "Validation rows:",
    len(X_validation),
)

Training rows: 364798
Training features: 56
Validation rows: 71256


In [64]:
historical_model_records = {
    str(record["model"]): record
    for record in model_selection_report[
        "models_evaluated"
    ]
}


historical_xgboost_records = {
    str(record["model"]): record
    for record in model_selection_report[
        "xgboost_configurations"
    ]
}


EXPECTED_HISTORICAL_METRICS = {
    "current_persistence": (
        historical_model_records[
            "current_persistence"
        ]
    ),
    "previous_day_persistence": (
        historical_model_records[
            "previous_day_persistence"
        ]
    ),
    "ridge": (
        historical_model_records[
            "ridge"
        ]
    ),
    "hist_gradient_boosting": (
        historical_model_records[
            "hist_gradient_boosting"
        ]
    ),
    "xgboost_shallower": (
        historical_xgboost_records[
            "xgboost_shallower"
        ]
    ),
}


expected_historical_metrics_df = (
    pd.DataFrame(
        [
            {
                "model": model_name,
                "mae": float(
                    metrics["mae"]
                ),
                "rmse": float(
                    metrics["rmse"]
                ),
                "r2": float(
                    metrics["r2"]
                ),
            }
            for model_name, metrics
            in EXPECTED_HISTORICAL_METRICS.items()
        ]
    )
)


display(
    expected_historical_metrics_df
)

,model,mae,rmse,r2
0,current_persistence,6.621121,10.506971,-0.161334
1,previous_day_persistence,7.942474,11.887120,-0.486467
2,ridge,9.779916,12.994571,-0.776340
3,hist_gradient_boosting,7.176154,9.827731,-0.016035
4,xgboost_shallower,7.080239,9.604153,0.029668


### **12D.1.** Persistence Controls

The two original persistence baselines are reproduced first.

Current-value persistence predicts every future PM2.5 value using the PM2.5
observed at the forecast reference time.

Previous-day persistence predicts using the PM2.5 value observed 24 hours
before the reference time.

These predictions require no model fitting and provide simple reference points
for the learned estimators.

In [65]:
current_persistence_predictions = (
    validation_df[
        "pm25_current"
    ]
    .astype(float)
    .to_numpy()
)

previous_day_persistence_predictions = (
    validation_df[
        "pm25_lag_24h"
    ]
    .astype(float)
    .to_numpy()
)


assert np.isfinite(
    current_persistence_predictions
).all()

assert np.isfinite(
    previous_day_persistence_predictions
).all()


current_persistence_metrics_12d = (
    calculate_regression_metrics(
        y_true=y_validation,
        y_pred=(
            current_persistence_predictions
        ),
    )
)

previous_day_persistence_metrics_12d = (
    calculate_regression_metrics(
        y_true=y_validation,
        y_pred=(
            previous_day_persistence_predictions
        ),
    )
)

In [66]:
persistence_reproduction_df = pd.DataFrame(
    [
        {
            "model": (
                "current_persistence"
            ),
            "expected_mae": float(
                EXPECTED_HISTORICAL_METRICS[
                    "current_persistence"
                ]["mae"]
            ),
            "reproduced_mae": (
                current_persistence_metrics_12d[
                    "mae"
                ]
            ),
            "expected_rmse": float(
                EXPECTED_HISTORICAL_METRICS[
                    "current_persistence"
                ]["rmse"]
            ),
            "reproduced_rmse": (
                current_persistence_metrics_12d[
                    "rmse"
                ]
            ),
            "expected_r2": float(
                EXPECTED_HISTORICAL_METRICS[
                    "current_persistence"
                ]["r2"]
            ),
            "reproduced_r2": (
                current_persistence_metrics_12d[
                    "r2"
                ]
            ),
        },
        {
            "model": (
                "previous_day_persistence"
            ),
            "expected_mae": float(
                EXPECTED_HISTORICAL_METRICS[
                    "previous_day_persistence"
                ]["mae"]
            ),
            "reproduced_mae": (
                previous_day_persistence_metrics_12d[
                    "mae"
                ]
            ),
            "expected_rmse": float(
                EXPECTED_HISTORICAL_METRICS[
                    "previous_day_persistence"
                ]["rmse"]
            ),
            "reproduced_rmse": (
                previous_day_persistence_metrics_12d[
                    "rmse"
                ]
            ),
            "expected_r2": float(
                EXPECTED_HISTORICAL_METRICS[
                    "previous_day_persistence"
                ]["r2"]
            ),
            "reproduced_r2": (
                previous_day_persistence_metrics_12d[
                    "r2"
                ]
            ),
        },
    ]
)


display(
    persistence_reproduction_df
)

,model,expected_mae,reproduced_mae,expected_rmse,reproduced_rmse,expected_r2,reproduced_r2
0,current_persistence,6.621121,6.621121,10.506971,10.506971,-0.161334,-0.161334
1,previous_day_persistence,7.942474,7.942474,11.887120,11.887120,-0.486467,-0.486467


In [67]:
for (
    model_name,
    reproduced_metrics,
) in (
    (
        "current_persistence",
        current_persistence_metrics_12d,
    ),
    (
        "previous_day_persistence",
        previous_day_persistence_metrics_12d,
    ),
):
    expected_metrics = (
        EXPECTED_HISTORICAL_METRICS[
            model_name
        ]
    )

    for metric_name in (
        "mae",
        "rmse",
        "r2",
    ):
        np.testing.assert_allclose(
            reproduced_metrics[
                metric_name
            ],
            float(
                expected_metrics[
                    metric_name
                ]
            ),
            rtol=0.0,
            atol=METRIC_ABSOLUTE_TOLERANCE,
            err_msg=(
                "Historical persistence "
                "metric did not reproduce: "
                f"{model_name}/{metric_name}"
            ),
        )


print(
    "Historical persistence controls "
    "reproduced successfully."
)

Historical persistence controls reproduced successfully.


### **12D.2.** Ridge Regression Control

The original Ridge Regression control is now reproduced using the same Phase 3
configuration.

The model is intentionally not tuned.

It uses:

1. `StandardScaler` fitted only on the training split.
2. `Ridge(alpha=1.0)` fitted on the scaled training features.

The full 1–72 hour training and validation datasets are used because the goal
of this subphase is to reproduce the historical Phase 3 experiment, not yet to
create a new horizon-specialized challenger.

In [68]:
HISTORICAL_RIDGE_ALPHA = 1.0


historical_ridge_model = Pipeline(
    steps=[
        (
            "scaler",
            StandardScaler(),
        ),
        (
            "model",
            Ridge(
                alpha=(
                    HISTORICAL_RIDGE_ALPHA
                ),
                solver="auto",
            ),
        ),
    ]
)


historical_ridge_model

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('scaler', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
,"alpha alpha: float or array-like of shape (n_targets,), default=1.0Constant that multiplies the L2 term, controlling regularizationstrength. `alpha` must be a non-negative float i.e. in `[0, inf)`.When `alpha = 0`, the objective is equivalent to ordinary leastsquares, solved by the :class:`LinearRegression` object. For numericalreasons, using `alpha = 0` with the `Ridge` object is not advised.Instead, you should use the :class:`LinearRegression` object.If an array is passed, penalties are assumed to be specific to thetargets. Hence they must correspond in number.See :ref:`sphx_glr_auto_examples_linear_model_plot_ridge_coeffs.py`for an illustration of the effect of alpha on the model coefficients.",1.0
,"fit_intercept fit_intercept: bool, default=TrueWhether to fit the intercept for this model. If setto false, no intercept will be used in calculations(i.e. ``X`` and ``y`` are expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"max_iter max_iter: int, default=NoneMaximum number of iterations for conjugate gradient solver.For 'sparse_cg' and 'lsqr' solvers, the default value is determinedby scipy.sparse.linalg. For 'sag' solver, the default value is 1000.For 'lbfgs' solver, the default value is 15000.",None


In [69]:
ridge_training_start = perf_counter()

historical_ridge_model.fit(
    X_train,
    y_train,
)

ridge_training_seconds_12d = (
    perf_counter()
    - ridge_training_start
)


print(
    "Historical Ridge training time:",
    f"{ridge_training_seconds_12d:.3f}",
    "seconds",
)

Historical Ridge training time: 3.076 seconds


In [70]:
ridge_raw_prediction_start = (
    perf_counter()
)

ridge_raw_validation_predictions = (
    historical_ridge_model.predict(
        X_validation
    )
)

ridge_raw_prediction_seconds = (
    perf_counter()
    - ridge_raw_prediction_start
)


assert np.isfinite(
    ridge_raw_validation_predictions
).all()


ridge_historical_metrics_12d = (
    calculate_regression_metrics(
        y_true=y_validation,
        y_pred=(
            ridge_raw_validation_predictions
        ),
    )
)


display(
    pd.Series(
        ridge_historical_metrics_12d,
        name="reproduced_ridge",
    ).to_frame()
)

,reproduced_ridge
mae,9.779916
rmse,12.994571
r2,-0.776340


In [71]:
ridge_expected_metrics = (
    EXPECTED_HISTORICAL_METRICS[
        "ridge"
    ]
)


ridge_reproduction_df = pd.DataFrame(
    [
        {
            "metric": metric_name,
            "expected": float(
                ridge_expected_metrics[
                    metric_name
                ]
            ),
            "reproduced": (
                ridge_historical_metrics_12d[
                    metric_name
                ]
            ),
            "absolute_difference": abs(
                ridge_historical_metrics_12d[
                    metric_name
                ]
                - float(
                    ridge_expected_metrics[
                        metric_name
                    ]
                )
            ),
        }
        for metric_name in (
            "mae",
            "rmse",
            "r2",
        )
    ]
)


display(
    ridge_reproduction_df
)

,metric,expected,reproduced,absolute_difference
0,mae,9.779916,9.779916,6.039613e-14
1,rmse,12.994571,12.994571,1.207923e-13
2,r2,-0.776340,-0.776340,3.330669e-14


In [72]:
for metric_name in (
    "mae",
    "rmse",
    "r2",
):
    np.testing.assert_allclose(
        ridge_historical_metrics_12d[
            metric_name
        ],
        float(
            ridge_expected_metrics[
                metric_name
            ]
        ),
        rtol=0.0,
        atol=1e-5,
        err_msg=(
            "Historical Ridge metric "
            f"did not reproduce: {metric_name}"
        ),
    )


print(
    "Historical Ridge control "
    "reproduced successfully."
)

Historical Ridge control reproduced successfully.


### **12D.3.** Histogram Gradient Boosting Control

Histogram Gradient Boosting was the strongest non-XGBoost learned estimator in
the original Phase 3 comparison.

This subphase reproduces the original configuration without tuning it.

As with Ridge, the estimator is trained across the historical 1–72 hour
training rows because this is a reproduction control. Horizon-specialized
training will be introduced later as a genuinely new experiment.

In [73]:
HISTORICAL_RANDOM_SEED = 42


historical_histgb_model = (
    HistGradientBoostingRegressor(
        loss="squared_error",
        learning_rate=0.05,
        max_iter=300,
        max_leaf_nodes=31,
        min_samples_leaf=30,
        l2_regularization=1.0,
        early_stopping=False,
        random_state=(
            HISTORICAL_RANDOM_SEED
        ),
    )
)


historical_histgb_model

,"learning_rate learning_rate: float, default=0.1The learning rate, also known as *shrinkage*. This is used as amultiplicative factor for the leaves values. Use ``1`` for noshrinkage.",0.05
,"max_iter max_iter: int, default=100The maximum number of iterations of the boosting process, i.e. themaximum number of trees.",300
,"min_samples_leaf min_samples_leaf: int, default=20The minimum number of samples per leaf. For small datasets with lessthan a few hundred samples, it is recommended to lower this valuesince only very shallow trees would be built.",30
,"l2_regularization l2_regularization: float, default=0The L2 regularization parameter penalizing leaves with small hessians.Use ``0`` for no regularization (default).",1.0
,"early_stopping early_stopping: 'auto' or bool, default='auto'If 'auto', early stopping is enabled if the sample size is larger than10000 or if `X_val` and `y_val` are passed to `fit`. If True, early stoppingis enabled, otherwise early stopping is disabled... versionadded:: 0.23",False
,"random_state random_state: int, RandomState instance or None, default=NonePseudo-random number generator to control the subsampling in thebinning process, and the train/validation data split if early stoppingis enabled.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",42
,"loss loss: {'squared_error', 'absolute_error', 'gamma', 'poisson', 'quantile'}, default='squared_error'The loss function to use in the boosting process. Note that the""squared error"", ""gamma"" and ""poisson"" losses actually implement""half least squares loss"", ""half gamma deviance"" and ""half poissondeviance"" to simplify the computation of the gradient. Furthermore,""gamma"" and ""poisson"" losses internally use a log-link, ""gamma""requires ``y > 0`` and ""poisson"" requires ``y >= 0``.""quantile"" uses the pinball loss... versionchanged:: 0.23 Added option 'poisson'... versionchanged:: 1.1 Added option 'quantile'... versionchanged:: 1.3 Added option 'gamma'.",'squared_error'
,"quantile quantile: float, default=NoneIf loss is ""quantile"", this parameter specifies which quantile to be estimatedand must be between 0 and 1.",None
,"max_leaf_nodes max_leaf_nodes: int or None, default=31The maximum number of leaves for each tree. Must be strictly greaterthan 1. If None, there is no maximum limit.",31
,"max_depth max_depth: int or None, default=NoneThe maximum depth of each tree. The depth of a tree is the number ofedges to go from the root to the deepest leaf.Depth isn't constrained by default.",None
,"max_features max_features: float, default=1.0Proportion of randomly chosen features in each and every node split.This is a form of regularization, smaller values make the trees weakerlearners and might prevent overfitting.If interaction constraints from `interaction_cst` are present, only allowedfeatures are taken into account for the subsampling... versionadded:: 1.4",1.0


In [74]:
histgb_training_start = (
    perf_counter()
)

historical_histgb_model.fit(
    X_train,
    y_train,
)

histgb_training_seconds_12d = (
    perf_counter()
    - histgb_training_start
)


print(
    "Historical HistGB training time:",
    f"{histgb_training_seconds_12d:.3f}",
    "seconds",
)

print(
    "Iterations completed:",
    historical_histgb_model.n_iter_,
)

Historical HistGB training time: 34.015 seconds
Iterations completed: 300


In [75]:
histgb_raw_prediction_start = (
    perf_counter()
)

histgb_raw_validation_predictions = (
    historical_histgb_model.predict(
        X_validation
    )
)

histgb_raw_prediction_seconds = (
    perf_counter()
    - histgb_raw_prediction_start
)


assert np.isfinite(
    histgb_raw_validation_predictions
).all()


histgb_historical_metrics_12d = (
    calculate_regression_metrics(
        y_true=y_validation,
        y_pred=(
            histgb_raw_validation_predictions
        ),
    )
)


display(
    pd.Series(
        histgb_historical_metrics_12d,
        name="reproduced_histgb",
    ).to_frame()
)

,reproduced_histgb
mae,7.176154
rmse,9.827731
r2,-0.016035


In [76]:
histgb_expected_metrics = (
    EXPECTED_HISTORICAL_METRICS[
        "hist_gradient_boosting"
    ]
)


histgb_reproduction_df = pd.DataFrame(
    [
        {
            "metric": metric_name,
            "expected": float(
                histgb_expected_metrics[
                    metric_name
                ]
            ),
            "reproduced": (
                histgb_historical_metrics_12d[
                    metric_name
                ]
            ),
            "absolute_difference": abs(
                histgb_historical_metrics_12d[
                    metric_name
                ]
                - float(
                    histgb_expected_metrics[
                        metric_name
                    ]
                )
            ),
        }
        for metric_name in (
            "mae",
            "rmse",
            "r2",
        )
    ]
)


display(
    histgb_reproduction_df
)

,metric,expected,reproduced,absolute_difference
0,mae,7.176154,7.176154,1.243450e-14
1,rmse,9.827731,9.827731,8.881784e-15
2,r2,-0.016035,-0.016035,1.776357e-15


In [77]:
for metric_name in (
    "mae",
    "rmse",
    "r2",
):
    np.testing.assert_allclose(
        histgb_historical_metrics_12d[
            metric_name
        ],
        float(
            histgb_expected_metrics[
                metric_name
            ]
        ),
        rtol=0.0,
        atol=1e-5,
        err_msg=(
            "Historical HistGB metric "
            f"did not reproduce: {metric_name}"
        ),
    )


print(
    "Historical HistGradientBoosting "
    "control reproduced successfully."
)

Historical HistGradientBoosting control reproduced successfully.


### **12D.4.** Saved XGBoost Raw-Model Control

The saved production estimator is now evaluated without hybrid persistence
routing.

This reproduces the historical learned-model view of
`xgboost_shallower` across the complete validation split.

The model is not retrained. The immutable production artifact loaded in Phase
12B is reused.

In [78]:
xgboost_raw_prediction_start = (
    perf_counter()
)

xgboost_raw_validation_predictions = (
    production_model.predict(
        X_validation
    )
)

xgboost_raw_prediction_seconds_12d = (
    perf_counter()
    - xgboost_raw_prediction_start
)


assert np.isfinite(
    xgboost_raw_validation_predictions
).all()


xgboost_historical_metrics_12d = (
    calculate_regression_metrics(
        y_true=y_validation,
        y_pred=(
            xgboost_raw_validation_predictions
        ),
    )
)


display(
    pd.Series(
        xgboost_historical_metrics_12d,
        name="saved_xgboost_shallower",
    ).to_frame()
)

,saved_xgboost_shallower
mae,7.080239
rmse,9.604153
r2,0.029668


In [79]:
xgboost_expected_metrics = (
    EXPECTED_HISTORICAL_METRICS[
        "xgboost_shallower"
    ]
)


xgboost_raw_reproduction_df = pd.DataFrame(
    [
        {
            "metric": metric_name,
            "expected": float(
                xgboost_expected_metrics[
                    metric_name
                ]
            ),
            "reproduced": (
                xgboost_historical_metrics_12d[
                    metric_name
                ]
            ),
            "absolute_difference": abs(
                xgboost_historical_metrics_12d[
                    metric_name
                ]
                - float(
                    xgboost_expected_metrics[
                        metric_name
                    ]
                )
            ),
        }
        for metric_name in (
            "mae",
            "rmse",
            "r2",
        )
    ]
)


display(
    xgboost_raw_reproduction_df
)


for metric_name in (
    "mae",
    "rmse",
    "r2",
):
    np.testing.assert_allclose(
        xgboost_historical_metrics_12d[
            metric_name
        ],
        float(
            xgboost_expected_metrics[
                metric_name
            ]
        ),
        rtol=0.0,
        atol=METRIC_ABSOLUTE_TOLERANCE,
    )


print(
    "Saved xgboost_shallower raw "
    "validation metrics reproduced."
)

,metric,expected,reproduced,absolute_difference
0,mae,7.080239,7.080239,0.000000e+00
1,rmse,9.604153,9.604153,0.000000e+00
2,r2,0.029668,0.029668,2.220446e-16


Saved xgboost_shallower raw validation metrics reproduced.


In [80]:
historical_control_comparison_df = (
    pd.DataFrame(
        [
            {
                "model": (
                    "xgboost_shallower"
                ),
                **xgboost_historical_metrics_12d,
            },
            {
                "model": (
                    "hist_gradient_boosting"
                ),
                **histgb_historical_metrics_12d,
            },
            {
                "model": (
                    "current_persistence"
                ),
                **current_persistence_metrics_12d,
            },
            {
                "model": (
                    "previous_day_persistence"
                ),
                **previous_day_persistence_metrics_12d,
            },
            {
                "model": "ridge",
                **ridge_historical_metrics_12d,
            },
        ]
    )
    .sort_values(
        "rmse"
    )
    .reset_index(
        drop=True
    )
)


display(
    historical_control_comparison_df
)

,model,mae,rmse,r2
0,xgboost_shallower,7.080239,9.604153,0.029668
1,hist_gradient_boosting,7.176154,9.827731,-0.016035
2,current_persistence,6.621121,10.506971,-0.161334
3,previous_day_persistence,7.942474,11.887120,-0.486467
4,ridge,9.779916,12.994571,-0.776340


In [81]:
assert (
    historical_control_comparison_df[
        "model"
    ].tolist()
    == [
        "xgboost_shallower",
        "hist_gradient_boosting",
        "current_persistence",
        "previous_day_persistence",
        "ridge",
    ]
)


print(
    "Historical Phase 3 control ordering "
    "has been reproduced."
)

Historical Phase 3 control ordering has been reproduced.


### **12D.5.** Historical Models Under the Current Hybrid Strategy

The reproduced Ridge and Histogram Gradient Boosting estimators are now
evaluated through the current production routing architecture.

For each model:

- hours 1–12 use current-value persistence
- hours 13–72 use the fitted estimator

This does not make either estimator a new challenger. Their training remains
identical to the original Phase 3 configuration.

The purpose is to establish how those historical controls compare when judged
under the same hybrid contract that all later challengers will use.

In [82]:
(
    ridge_hybrid_raw_predictions,
    ridge_hybrid_predictions,
    ridge_hybrid_prediction_seconds,
) = generate_timed_hybrid_predictions(
    dataframe=validation_df,
    model=historical_ridge_model,
    feature_columns=MODEL_FEATURE_COLUMNS,
    persistence_max_horizon=(
        PERSISTENCE_MAX_HORIZON
    ),
)


ridge_model_size_bytes = (
    measure_serialized_model_size_bytes(
        historical_ridge_model
    )
)


ridge_hybrid_result = (
    evaluate_candidate_predictions(
        candidate_name=(
            "historical_ridge_hybrid"
        ),
        model_family="Ridge",
        training_scope=(
            "historical 1-72h control"
        ),
        dataframe=validation_df,
        raw_predictions=(
            ridge_hybrid_raw_predictions
        ),
        operational_predictions=(
            ridge_hybrid_predictions
        ),
        training_seconds=(
            ridge_training_seconds_12d
        ),
        prediction_seconds=(
            ridge_hybrid_prediction_seconds
        ),
        model_size_bytes=(
            ridge_model_size_bytes
        ),
        parameters={
            "alpha": (
                HISTORICAL_RIDGE_ALPHA
            ),
            "scaled": True,
        },
    )
)


benchmark_results[
    "historical_ridge_hybrid"
] = ridge_hybrid_result

In [83]:
(
    histgb_hybrid_raw_predictions,
    histgb_hybrid_predictions,
    histgb_hybrid_prediction_seconds,
) = generate_timed_hybrid_predictions(
    dataframe=validation_df,
    model=historical_histgb_model,
    feature_columns=MODEL_FEATURE_COLUMNS,
    persistence_max_horizon=(
        PERSISTENCE_MAX_HORIZON
    ),
)


histgb_model_size_bytes = (
    measure_serialized_model_size_bytes(
        historical_histgb_model
    )
)


histgb_hybrid_result = (
    evaluate_candidate_predictions(
        candidate_name=(
            "historical_histgb_hybrid"
        ),
        model_family=(
            "HistGradientBoostingRegressor"
        ),
        training_scope=(
            "historical 1-72h control"
        ),
        dataframe=validation_df,
        raw_predictions=(
            histgb_hybrid_raw_predictions
        ),
        operational_predictions=(
            histgb_hybrid_predictions
        ),
        training_seconds=(
            histgb_training_seconds_12d
        ),
        prediction_seconds=(
            histgb_hybrid_prediction_seconds
        ),
        model_size_bytes=(
            histgb_model_size_bytes
        ),
        parameters={
            "loss": "squared_error",
            "learning_rate": 0.05,
            "max_iter": 300,
            "max_leaf_nodes": 31,
            "min_samples_leaf": 30,
            "l2_regularization": 1.0,
            "early_stopping": False,
            "random_state": 42,
        },
    )
)


benchmark_results[
    "historical_histgb_hybrid"
] = histgb_hybrid_result

In [84]:
historical_hybrid_control_df = (
    pd.DataFrame(
        [
            build_leaderboard_row(
                benchmark_results[
                    "production_champion_v1"
                ]
            ),
            build_leaderboard_row(
                benchmark_results[
                    "historical_histgb_hybrid"
                ]
            ),
            build_leaderboard_row(
                benchmark_results[
                    "historical_ridge_hybrid"
                ]
            ),
        ]
    )
    .sort_values(
        "hybrid_rmse"
    )
    .reset_index(
        drop=True
    )
)


display(
    historical_hybrid_control_df
)

,candidate,model_family,training_scope,hybrid_mae,hybrid_rmse,hybrid_r2,learned_13_72_mae,learned_13_72_rmse,learned_13_72_r2,negative_raw_predictions,clipped_predictions,training_seconds,prediction_seconds,model_size_mib
0,production_champion_v1,XGBRegressor,historical production artifact,6.695642,9.419553,0.066611,7.178338,9.713671,-0.009030,0,0,NaN,0.269260,0.645863
1,historical_histgb_hybrid,HistGradientBoostingRegressor,historical 1-72h control,6.762337,9.579777,0.034588,7.258972,9.901305,-0.048389,0,0,34.014544,1.166313,1.096182
2,historical_ridge_hybrid,Ridge,historical 1-72h control,8.162262,11.310933,-0.345857,8.951510,11.906307,-0.515973,9848,9848,3.075756,0.313520,0.003968


In [85]:
champion_hybrid_rmse = (
    benchmark_results[
        "production_champion_v1"
    ][
        "overall_metrics"
    ]["rmse"]
)

champion_hybrid_mae = (
    benchmark_results[
        "production_champion_v1"
    ][
        "overall_metrics"
    ]["mae"]
)


historical_hybrid_control_df[
    "rmse_improvement_vs_champion_pct"
] = historical_hybrid_control_df[
    "hybrid_rmse"
].map(
    lambda value: (
        calculate_improvement_percent(
            champion_value=(
                champion_hybrid_rmse
            ),
            candidate_value=float(
                value
            ),
        )
    )
)


historical_hybrid_control_df[
    "mae_improvement_vs_champion_pct"
] = historical_hybrid_control_df[
    "hybrid_mae"
].map(
    lambda value: (
        calculate_improvement_percent(
            champion_value=(
                champion_hybrid_mae
            ),
            candidate_value=float(
                value
            ),
        )
    )
)


display(
    historical_hybrid_control_df[
        [
            "candidate",
            "model_family",
            "hybrid_mae",
            "hybrid_rmse",
            "hybrid_r2",
            "learned_13_72_mae",
            "learned_13_72_rmse",
            "rmse_improvement_vs_champion_pct",
            "mae_improvement_vs_champion_pct",
            "negative_raw_predictions",
            "training_seconds",
            "prediction_seconds",
            "model_size_mib",
        ]
    ]
)

,candidate,model_family,hybrid_mae,hybrid_rmse,hybrid_r2,learned_13_72_mae,learned_13_72_rmse,rmse_improvement_vs_champion_pct,mae_improvement_vs_champion_pct,negative_raw_predictions,training_seconds,prediction_seconds,model_size_mib
0,production_champion_v1,XGBRegressor,6.695642,9.419553,0.066611,7.178338,9.713671,0.000000,0.000000,0,NaN,0.269260,0.645863
1,historical_histgb_hybrid,HistGradientBoostingRegressor,6.762337,9.579777,0.034588,7.258972,9.901305,-1.700974,-0.996086,0,34.014544,1.166313,1.096182
2,historical_ridge_hybrid,Ridge,8.162262,11.310933,-0.345857,8.951510,11.906307,-20.079305,-21.904098,9848,3.075756,0.313520,0.003968


In [86]:
historical_horizon_group_comparison_df = (
    pd.concat(
        [
            benchmark_results[
                "production_champion_v1"
            ][
                "horizon_group_metrics"
            ],
            benchmark_results[
                "historical_histgb_hybrid"
            ][
                "horizon_group_metrics"
            ],
            benchmark_results[
                "historical_ridge_hybrid"
            ][
                "horizon_group_metrics"
            ],
        ],
        ignore_index=True,
    )
    .sort_values(
        [
            "horizon_min",
            "rmse",
        ]
    )
    .reset_index(
        drop=True
    )
)


display(
    historical_horizon_group_comparison_df
)

,model,horizon_group,horizon_min,horizon_max,rows,mae,rmse,r2
0,production_champion_v1,1-6h,1,6,6190,3.648142,6.990142,0.516136
1,historical_histgb_hybrid,1-6h,1,6,6190,3.648142,6.990142,0.516136
2,historical_ridge_hybrid,1-6h,1,6,6190,3.648142,6.990142,0.516136
3,production_champion_v1,7-12h,7,12,6129,5.131832,8.653139,0.268671
4,historical_histgb_hybrid,7-12h,7,12,6129,5.131832,8.653139,0.268671
5,historical_ridge_hybrid,7-12h,7,12,6129,5.131832,8.653139,0.268671
6,production_champion_v1,13-24h,13,24,12163,7.117727,9.692964,0.091642
7,historical_histgb_hybrid,13-24h,13,24,12163,7.160805,9.813267,0.068954
8,historical_ridge_hybrid,13-24h,13,24,12163,8.831251,11.769248,-0.339187
9,production_champion_v1,25-48h,25,48,23709,7.168849,9.669828,-0.050194


In [87]:
for group_name in (
    "1-6h",
    "7-12h",
):
    group_df = (
        historical_horizon_group_comparison_df.loc[
            historical_horizon_group_comparison_df[
                "horizon_group"
            ].eq(group_name)
        ]
    )

    for metric_name in (
        "mae",
        "rmse",
        "r2",
    ):
        metric_values = (
            group_df[
                metric_name
            ]
            .astype(float)
            .to_numpy()
        )

        np.testing.assert_allclose(
            metric_values,
            np.repeat(
                metric_values[0],
                len(metric_values),
            ),
            rtol=0.0,
            atol=1e-12,
        )


print(
    "All hybrid controls are identical "
    "for persistence-routed horizons 1–12."
)

All hybrid controls are identical for persistence-routed horizons 1–12.


In [88]:
historical_control_summary = {
    "status": "PASSED",
    "historical_order_reproduced": True,
    "persistence_controls_reproduced": True,
    "ridge_control_reproduced": True,
    "histgb_control_reproduced": True,
    "xgboost_raw_control_reproduced": True,
    "hybrid_short_horizon_equivalence": True,
    "benchmark_result_count": len(
        benchmark_results
    ),
    "registered_hybrid_controls": [
        "production_champion_v1",
        "historical_histgb_hybrid",
        "historical_ridge_hybrid",
    ],
}


display(
    pd.Series(
        historical_control_summary,
        name="value",
    ).to_frame()
)


print(
    "Phase 12D PASSED — historical "
    "controls are consistent with Phase 3."
)

,value
status,PASSED
historical_order_reproduced,True
persistence_controls_reproduced,True
ridge_control_reproduced,True
histgb_control_reproduced,True
xgboost_raw_control_reproduced,True
hybrid_short_horizon_equivalence,True
benchmark_result_count,3
registered_hybrid_controls,"[production_champion_v1, historical_histgb_hyb..."


Phase 12D PASSED — historical controls are consistent with Phase 3.
